# Treinamento de Modelo Preditivo - Risco de Defasagem Escolar

Este notebook apresenta o desenvolvimento de um modelo preditivo para identificar alunos com **Risco de Defasagem Escolar** (definido como proficiência < 720).
Utilizaremos o **BigQuery ML (BQML)** para treinar e avaliar os modelos diretamente no BigQuery.

## Objetivos:
1. Analisar a base de dados preparada na camada Ouro (`gold_dataset_ml_proficiencia_alunos`).
2. Treinar um modelo de Regressão Logística como baseline.
3. Treinar um modelo de Boosted Trees (XGBoost) para comparação.
4. Avaliar os modelos e escolher o melhor para produção.
5. Analisar a importância das features.

In [ ]:
# Inicialização do BigFrames e carregamento das extensões
import bigframes
import bigframes.pandas as bpd

# Configurando o projeto padrão
bpd.options.bigquery.project = "vanehay"
bpd.options.bigquery.location = "us-central1"

%load_ext bigframes
print("BigFrames inicializado com sucesso. Versão:", bigframes.__version__)

## 1. Explorando os Dados

Nesta etapa, vamos visualizar as primeiras linhas do dataset de ML para entender a estrutura e os tipos de dados.
Também verificaremos se existem valores nulos ou inconsistentes nas variáveis que usaremos como features.

As features disponíveis são:
*   `ano`: Ano da avaliação.
*   `id_municipio`: Código IBGE do município (territorial).
*   `rede`: Rede de ensino (categórica).
*   `serie`: Série escolar (categórica).
*   `preenchimento_caderno`: Status do preenchimento.
*   `peso_aluno`: Peso amostral.
*   `meta_municipio_2024`: Meta de alfabetização.

O Target é:
*   `target_risco_defasagem_bool`: 1 se proficiência < 720, 0 caso contrário.

In [ ]:
%%bqsql df_preview
SELECT * FROM `vanehay.1IAST_Fase2.gold_dataset_ml_proficiencia_alunos` LIMIT 10

In [ ]:
df_preview

### Distribuição da Variável Alvo (Target)

Vamos verificar se há desbalanceamento de classes na variável `target_risco_defasagem_bool`.
O desbalanceamento pode afetar a performance do modelo, especialmente para métricas como F1-Score e Recall.

In [ ]:
%%bqsql df_target_dist
SELECT 
  target_risco_defasagem_bool, 
  COUNT(*) as total_alunos,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentagem
FROM `vanehay.1IAST_Fase2.gold_dataset_ml_proficiencia_alunos`
GROUP BY target_risco_defasagem_bool

In [ ]:
df_target_dist

#### Análise da Distribuição do Target

A análise da distribuição mostra:
*   **Classe 0 (Sem Risco de Defasagem)**: 74.88% (~2.51 milhões de alunos)
*   **Classe 1 (Em Risco de Defasagem)**: 25.12% (~842 mil alunos)

**Conclusão**: Há um desbalanceamento moderado (proporção de aproximadamente 3:1). 
Ao avaliar os modelos, devemos focar em métricas como **Precision**, **Recall** e **F1-Score** para a **Classe 1** (alunos em risco), pois identificar corretamente esses alunos é o objetivo principal do negócio. A acurácia simples pode ser enganosa neste cenário.

## 2. Modelo Baseline: Regressão Logística

Como primeiro passo, vamos treinar um modelo de **Regressão Logística**. É um modelo simples, rápido de treinar e que serve como um excelente baseline (ponto de partida).

Usaremos a opção `auto_class_weights=TRUE` para que o BigQuery ML ajuste automaticamente os pesos das classes para lidar com o desbalanceamento que identificamos (75/25).

As features utilizadas serão:
*   `rede` (categórica)
*   `serie` (categórica)
*   `preenchimento_caderno` (categórica)
*   `meta_municipio_2024` (numérica)

O target é `target_risco_defasagem_bool`.

In [ ]:
%%bqsql
CREATE OR REPLACE MODEL `vanehay.1IAST_Fase2.model_risco_defasagem_baseline`
OPTIONS(
  model_type="LOGISTIC_REG",
  input_label_cols=["target_risco_defasagem_bool"],
  auto_class_weights=TRUE
) AS
SELECT
  rede,
  serie,
  preenchimento_caderno,
  meta_municipio_2024,
  target_risco_defasagem_bool
FROM
  `vanehay.1IAST_Fase2.gold_dataset_ml_proficiencia_alunos`

### Avaliação do Baseline

Vamos avaliar a performance do modelo baseline utilizando as métricas padrão para classificação binária fornecidas pelo BQML.
Focaremos principalmente no F1-Score e Recall para a classe de risco.

In [ ]:
%%bqsql df_baseline_eval
SELECT * FROM ML.EVALUATE(MODEL `vanehay.1IAST_Fase2.model_risco_defasagem_baseline`)

In [ ]:
df_baseline_eval

#### Análise dos Resultados do Baseline

O modelo de Regressão Logística apresentou os seguintes resultados na base de avaliação:
*   **Acurácia**: 57.88%
*   **Recall (Sensibilidade)**: 64.91% (consegue identificar ~65% dos alunos que realmente estão em risco)
*   **Precision (Precisão)**: 33.19% (quando o modelo aponta risco, ele está correto em ~33% dos casos, gerando muitos falsos positivos)
*   **F1-Score**: 43.92%
*   **ROC AUC**: 0.6378 (indica um poder preditivo moderado, acima do aleatório que seria 0.5)

**Conclusão**: O modelo baseline tem uma capacidade razoável de capturar os alunos em risco (Recall), mas à custa de muitos alarmes falsos (baixa Precisão). Isso é comum em modelos lineares simples com poucas features. 

Vamos agora tentar um modelo mais robusto: **Boosted Trees (XGBoost)**, que pode capturar relações não-lineares entre as features.

## 3. Modelo Avançado: Boosted Trees (XGBoost)

Agora vamos treinar um modelo de **Boosted Trees (XGBoost)**. Este modelo baseado em árvores de decisão costuma apresentar performance superior para dados estruturados, pois consegue mapear interações complexas entre as variáveis (por exemplo, como a combinação de certas redes de ensino com determinadas metas municipais afeta o risco).

Utilizaremos parâmetros padrão de treinamento do BigQuery ML para este tipo de modelo.

In [ ]:
%%bqsql
CREATE OR REPLACE MODEL `vanehay.1IAST_Fase2.model_risco_defasagem_xgboost`
OPTIONS(
  model_type="BOOSTED_TREE_CLASSIFIER",
  input_label_cols=["target_risco_defasagem_bool"],
  auto_class_weights=TRUE
) AS
SELECT
  rede,
  serie,
  preenchimento_caderno,
  meta_municipio_2024,
  target_risco_defasagem_bool
FROM
  `vanehay.1IAST_Fase2.gold_dataset_ml_proficiencia_alunos`

### Avaliação do XGBoost

Vamos avaliar a performance do modelo XGBoost e comparar com o baseline.
Esperamos ver uma melhora no F1-Score e na área sob a curva (ROC AUC).

In [ ]:
%%bqsql df_xgboost_eval
SELECT * FROM ML.EVALUATE(MODEL `vanehay.1IAST_Fase2.model_risco_defasagem_xgboost`)

In [ ]:
df_xgboost_eval

### Comparação dos Modelos

| Métrica | Regressão Logística (Baseline) | Boosted Trees (XGBoost) |
| :--- | :---: | :---: |
| **Acurácia** | 57.88% | **63.38%** |
| **Precision (Classe 1)** | 33.19% | **35.91%** |
| **Recall (Classe 1)** | **64.91%** | 56.23% |
| **F1-Score (Classe 1)** | **43.92%** | 43.83% |
| **ROC AUC** | 0.6378 | **0.6404** |
| **Log Loss** | 0.6866 | **0.6618** |

#### Análise da Comparação:
*   O **XGBoost** trouxe uma melhora na **Acurácia** (+5.5 p.p.) e na **Precisão** (+2.7 p.p.), além de reduzir o **Log Loss** (o que indica previsões mais calibradas).
*   No entanto, o **Recall** caiu significativamente (-8.7 p.p.), significando que o XGBoost deixa passar mais alunos em risco sem identificá-los.
*   O **F1-Score** permaneceu praticamente idêntico, mostrando que a melhora em precisão foi compensada pela perda em recall.

**Decisão de Negócio**: 
*   Se o objetivo é **maximizar a identificação** dos alunos em risco (mesmo gerando mais alarmes falsos para a equipe pedagógica atuar), o modelo de **Regressão Logística** ainda é preferível pelo maior Recall.
*   Se os recursos para intervenção forem escassos e precisarmos de **maior certeza** antes de agir, o **XGBoost** é ligeiramente melhor por ter maior Precisão.

Vamos analisar a importância das features no XGBoost para entender o que está guiando as decisões.

## 4. Importância das Features (XGBoost)

Vamos verificar quais variáveis têm maior impacto nas decisões do modelo XGBoost.
O BigQuery ML fornece a importância das features com base no ganho (gain) que cada feature traz para as divisões das árvores.

In [ ]:
%%bqsql df_feature_importance
SELECT * FROM ML.FEATURE_IMPORTANCE(MODEL `vanehay.1IAST_Fase2.model_risco_defasagem_xgboost`)

In [ ]:
df_feature_importance

#### Análise da Importância das Features

Os resultados mostram:
*   **`meta_municipio_2024`**: É a feature mais importante para o modelo (ganho de ~1283.7).
*   **`rede`**: Também tem impacto significativo (ganho de ~486.0).
*   **`serie`** e **`preenchimento_caderno`**: Têm importância **0.0**.

**Investigação**:
Ao analisar a distribuição dessas duas últimas variáveis, descobrimos que:
1.  A coluna `serie` possui apenas o valor `"2° ano do Ensino Fundamental"` para todos os 3.35 milhões de registros.
2.  A coluna `preenchimento_caderno` possui apenas o valor `"Prova Preenchida"` para todos os registros.

Como são valores constantes, elas não possuem poder preditivo e foram ignoradas pelo modelo.

### Próximos Passos Sugeridos:
Para melhorar a performance do modelo (que atualmente está com ROC AUC de ~0.64), precisamos de mais features. Uma excelente oportunidade é **incorporar os dados de Vulnerabilidade Social** que temos na tabela `silver_vulnerabilidade_social`, cruzando por `ano` e `id_municipio`.

Podemos adicionar variáveis como:
*   `qtd_familias_extrema_pobreza`
*   `qtd_familias_baixa_renda`
*   `qtd_familias_acima_meio_salario`

Isso nos dará um contexto socioeconômico do município do aluno, o que costuma ser fortemente correlacionado com o desempenho escolar.

## Resumo Executivo

### Data Analysis Key Findings
*   O dataset possui **3.354.661 registros** de alunos, todos pertencentes ao **2° ano do Ensino Fundamental**.
*   A taxa de alunos em risco de defasagem escolar (`target_risco_defasagem_bool = 1`) é de **25.12% (842.764 alunos)**, indicando um desbalanceamento moderado de 3:1.
*   O modelo baseline de **Regressão Logística** obteve:
    *   Acurácia: **57.88%**
    *   Recall: **64.91%**
    *   Precision: **33.19%**
    *   ROC AUC: **0.6378**
*   O modelo **XGBoost** obteve:
    *   Acurácia: **63.38%**
    *   Recall: **56.23%**
    *   Precision: **35.91%**
    *   ROC AUC: **0.6404**
*   As features `serie` e `preenchimento_caderno` apresentaram importância **zero** por possuírem valores constantes em toda a base.
*   A feature mais relevante no XGBoost foi `meta_municipio_2024` (ganho de **1283.74**), seguida por `rede` (**486.02**).

### Insights or Next Steps
*   **Escolha do Modelo**: Para uma abordagem preventiva (onde é pior deixar de atender um aluno em risco do que gerar um falso positivo), a **Regressão Logística** é recomendada pelo maior Recall (64.91%).
*   **Enriquecimento de Features**: É altamente recomendável enriquecer o dataset com dados socioeconômicos da tabela `silver_vulnerabilidade_social` (como extrema pobreza e baixa renda por município) para melhorar a capacidade preditiva do modelo, que atualmente está limitada a ~0.64 de ROC AUC devido ao baixo número de features úteis.